In [1]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv(
    "../data/processed/y_test.csv"
).squeeze("columns")

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (219, 92)
y_test shape: (219,)


In [5]:
best_ridge_pipeline = joblib.load(
    "../models/ridge_tuned_pipeline.pkl"
)

print("Tuned Ridge pipeline loaded successfully.")

Tuned Ridge pipeline loaded successfully.


In [6]:
# The pipeline performs preprocessing automatically.
y_pred_log = best_ridge_pipeline.predict(X_test)

print("Predictions generated:", len(y_pred_log))

Predictions generated: 219


In [7]:
log_rmse = np.sqrt(
    mean_squared_error(y_test, y_pred_log)
)

log_mae = mean_absolute_error(
    y_test, y_pred_log
)

log_r2 = r2_score(
    y_test, y_pred_log
)

print("LOG-SCALE TEST METRICS")
print(f"RMSE: {log_rmse:.4f}")
print(f"MAE:  {log_mae:.4f}")
print(f"R²:   {log_r2:.4f}")

LOG-SCALE TEST METRICS
RMSE: 0.1205
MAE:  0.0824
R²:   0.9147


In [8]:
y_test_dollar = np.expm1(y_test)
y_pred_dollar = np.expm1(y_pred_log)

print("Converted predictions to original price scale.")

Converted predictions to original price scale.


In [9]:
mae_dollar = mean_absolute_error(
    y_test_dollar,
    y_pred_dollar
)

rmse_dollar = np.sqrt(
    mean_squared_error(
        y_test_dollar,
        y_pred_dollar
    )
)

r2_dollar = r2_score(
    y_test_dollar,
    y_pred_dollar
)

print("ORIGINAL PRICE-SCALE TEST METRICS")
print(f"MAE:  ${mae_dollar:,.0f}")
print(f"RMSE: ${rmse_dollar:,.0f}")
print(f"R²:   {r2_dollar:.4f}")

ORIGINAL PRICE-SCALE TEST METRICS
MAE:  $13,908
RMSE: $19,154
R²:   0.9332


In [11]:
# Best CV RMSE obtained in 12_hyperparameter_tuning.ipynb
cv_rmse = 0.11421191908595789

final_report = pd.DataFrame({
    "Metric": [
        "CV RMSE",
        "Test RMSE (log scale)",
        "Test MAE (log scale)",
        "Test R2 (log scale)",
        "Test MAE (dollars)",
        "Test RMSE (dollars)",
        "Test R2 (dollars)"
    ],
    "Value": [
        cv_rmse,
        log_rmse,
        log_mae,
        log_r2,
        mae_dollar,
        rmse_dollar,
        r2_dollar
    ]
})

final_report.to_csv(
    "../models/final_model_report.csv",
    index=False
)

print(final_report)
print("Final report saved successfully.")

                  Metric         Value
0                CV RMSE      0.114212
1  Test RMSE (log scale)      0.120469
2   Test MAE (log scale)      0.082424
3    Test R2 (log scale)      0.914673
4     Test MAE (dollars)  13907.631861
5    Test RMSE (dollars)  19154.062704
6      Test R2 (dollars)      0.933214
Final report saved successfully.


In [12]:
import joblib

joblib.dump(
    best_ridge_pipeline,
    "../models/house_price_pipeline.pkl"
)

print("Final house-price pipeline saved successfully.")

Final house-price pipeline saved successfully.
